# Rock energy labeling: `energy_type_segment_quantile` + Plotly surfaces

Вход: `united.csv`

Этот notebook делает только необходимое:
1. создаёт energy-response признаки;
2. строит `formation_residual` и `hardness_score_smooth`;
3. размечает энергоёмкость через `energy_type_segment_quantile`;
4. сохраняет датасет с финальной разметкой;
5. строит интерактивные Plotly-поверхности `pressure_axis × pressure_rotation → speed` для каждой энергоёмкости.

Очистка датасета не выполняется: предполагается, что `united.csv` уже очищен.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json

import numpy as np
import pandas as pd

import plotly.graph_objects as go

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

try:
    import joblib
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

RANDOM_STATE = 42
EPS = 1e-6

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Загрузка данных

In [2]:
DATA_PATH = "../datasets/united.csv"

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Не хватает колонок: {missing}")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed"]].describe(percentiles=[.01, .05, .5, .95, .99]))

Loaded: ../datasets/united.csv
Shape: (415049, 6)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515


,pressure_axis,pressure_rotation,rotation,speed
count,415049.000000,415049.000000,415049.000000,415049.000000
mean,17473.667273,14134.146325,103.945315,0.013116
std,4682.982816,3243.523440,13.370361,0.006615
min,317.000000,784.000000,50.010000,0.001002
1%,3713.000000,6271.000000,64.980000,0.002755
5%,6645.000000,8246.000000,81.750000,0.005050
50%,18861.000000,14637.000000,103.158000,0.012120
95%,22343.000000,18758.600000,138.474000,0.024240
99%,23626.000000,20881.000000,139.020000,0.030300
max,24872.000000,26318.000000,139.578000,0.038957


## 2. Базовые признаки

In [3]:
df["dt"] = (
    df.groupby("well_id")["processing_time"]
      .diff()
      .dt.total_seconds()
)

df["dt"] = df["dt"].fillna(df["dt"].median())

df["total_pressure"] = df["pressure_axis"] + df["pressure_rotation"]

df["pressure_balance"] = (
    df["pressure_axis"] /
    (df["pressure_axis"] + df["pressure_rotation"] + EPS)
)

df["axis_over_rot_pressure"] = (
    df["pressure_axis"] /
    (df["pressure_rotation"] + EPS)
)

df["rot_pressure_over_axis"] = (
    df["pressure_rotation"] /
    (df["pressure_axis"] + EPS)
)

df["rotation_efficiency"] = (
    df["rotation"] /
    (df["pressure_rotation"] + EPS)
)

df["axis_x_rotation"] = df["pressure_axis"] * df["rotation"]
df["rot_pressure_x_rotation"] = df["pressure_rotation"] * df["rotation"]

df["energy_input_proxy"] = (
    df["pressure_axis"] +
    df["pressure_rotation"] * df["rotation"]
)

df["pseudo_mse"] = (
    df["energy_input_proxy"] /
    (df["speed"] + EPS)
)

df["drilling_efficiency"] = (
    df["speed"] /
    (df["energy_input_proxy"] + EPS)
)

df["log_energy_input_proxy"] = np.log1p(df["energy_input_proxy"])
df["log_pseudo_mse"] = np.log1p(df["pseudo_mse"])

display(df[[
    "dt",
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "rotation_efficiency",
    "pressure_balance",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

,dt,energy_input_proxy,pseudo_mse,drilling_efficiency,rotation_efficiency,pressure_balance
count,415049.000000,4.150490e+05,4.150490e+05,4.150490e+05,415049.000000,415049.000000
mean,7.270602,1.494500e+06,1.467727e+08,9.279041e-09,0.007803,0.545942
std,110.393457,4.009753e+05,9.155477e+07,5.931875e-09,0.002483,0.057506
min,0.089000,4.174649e+04,3.009300e+06,3.870720e-10,0.002104,0.013293
1%,0.229000,4.697380e+05,3.257078e+07,2.153750e-09,0.004789,0.320485
5%,0.464000,7.635049e+05,5.221720e+07,3.292781e-09,0.005407,0.428681
50%,5.135000,1.542361e+06,1.239899e+08,8.064516e-09,0.007111,0.558077
95%,10.891000,2.119089e+06,3.036335e+08,1.914947e-08,0.012153,0.607300
99%,23.689560,2.462635e+06,4.641533e+08,3.070111e-08,0.015835,0.629086
max,42583.603000,3.668551e+06,2.581497e+09,3.322895e-07,0.121481,0.933514


## 3. Rolling-признаки

In [4]:
def add_rolling_stats(data, cols, windows=(12, 30, 60), group_col="well_id"):
    out = data.copy()

    for col in cols:
        for w in windows:
            min_p = max(3, w // 3)

            out[f"{col}_roll_median_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).median())
            )

            out[f"{col}_roll_mean_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).mean())
            )

            out[f"{col}_roll_std_{w}"] = (
                out.groupby(group_col)[col]
                   .transform(lambda s: s.rolling(w, min_periods=min_p).std())
            )

    return out

rolling_cols = [
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "rotation_efficiency",
    "speed",
    "rotation",
    "pressure_balance",
]

df = add_rolling_stats(df, rolling_cols, windows=(12, 30, 60))

print("Rolling features added:", len([c for c in df.columns if "_roll_" in c]))

Rolling features added: 63


## 4. Expected speed и residual hardness

In [5]:
control_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "total_pressure",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "axis_x_rotation",
    "rot_pressure_x_rotation",
    "energy_input_proxy",
    "log_energy_input_proxy",
]

residual_df = df.dropna(subset=control_features + ["speed", "well_id"]).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(residual_df, groups=residual_df["well_id"]))

train_res = residual_df.iloc[train_idx].copy()
test_res = residual_df.iloc[test_idx].copy()

expected_speed_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ))
])

expected_speed_model.fit(train_res[control_features], train_res["speed"])

pred_test = expected_speed_model.predict(test_res[control_features])

mae = mean_absolute_error(test_res["speed"], pred_test)
rmse = root_mean_squared_error(test_res["speed"], pred_test)
r2 = r2_score(test_res["speed"], pred_test)

print(f"Expected-speed controls model | MAE={mae:.6f} | RMSE={rmse:.6f} | R2={r2:.4f}")

df["expected_speed_from_controls"] = expected_speed_model.predict(df[control_features])
df["formation_residual"] = df["speed"] - df["expected_speed_from_controls"]
df["relative_formation_residual"] = (
    df["formation_residual"] /
    (np.abs(df["expected_speed_from_controls"]) + EPS)
)

df = add_rolling_stats(
    df,
    cols=["formation_residual", "relative_formation_residual", "expected_speed_from_controls"],
    windows=(12, 30, 60),
)

display(df[[
    "speed",
    "expected_speed_from_controls",
    "formation_residual",
    "relative_formation_residual",
    "formation_residual_roll_median_60",
]].describe(percentiles=[.01, .05, .5, .95, .99]))

Expected-speed controls model | MAE=0.004750 | RMSE=0.006028 | R2=0.1725


,speed,expected_speed_from_controls,formation_residual,relative_formation_residual,formation_residual_roll_median_60
count,415049.000000,415049.000000,4.150490e+05,415049.000000,382635.000000
mean,0.013116,0.013116,3.750782e-07,-0.005738,-0.000321
std,0.006615,0.002594,5.960673e-03,0.459814,0.003772
min,0.001002,0.005363,-1.699507e-02,-0.936298,-0.011658
1%,0.002755,0.006579,-1.079520e-02,-0.777480,-0.007719
5%,0.005050,0.008102,-8.642517e-03,-0.618527,-0.005349
50%,0.012120,0.013441,-7.497199e-04,-0.061736,-0.001026
95%,0.024240,0.016886,1.057625e-02,0.811159,0.006899
99%,0.030300,0.017621,1.765780e-02,1.341649,0.010600
max,0.038957,0.018344,3.055098e-02,5.258323,0.020247


## 5. Continuous hardness score

In [6]:
def zscore(s):
    return (s - s.mean()) / (s.std() + EPS)

df["hardness_score"] = (
    zscore(df["log_pseudo_mse"])
    - zscore(df["drilling_efficiency"])
    - zscore(df["formation_residual"])
)

df["hardness_score_smooth"] = (
    zscore(np.log1p(df["pseudo_mse_roll_median_60"]))
    - zscore(df["drilling_efficiency_roll_median_60"])
    - zscore(df["formation_residual_roll_median_60"])
)

display(df[["hardness_score", "hardness_score_smooth"]].describe(percentiles=[.01, .05, .5, .95, .99]))

,hardness_score,hardness_score_smooth
count,4.150490e+05,3.826350e+05
mean,-4.329998e-15,4.278465e-15
std,1.929089e+00,1.898597e+00
min,-1.016414e+01,-1.023758e+01
1%,-5.092313e+00,-5.313689e+00
5%,-3.279611e+00,-3.437390e+00
50%,8.414619e-02,2.413973e-01
95%,2.963625e+00,2.752807e+00
99%,3.862814e+00,3.826881e+00
max,8.309555e+00,6.586316e+00


## 6. Финальная разметка энергоёмкости: `energy_type_segment_quantile`

In [7]:
energy_labels_4 = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

SEGMENT_SIZE = 60

segment_df = df.dropna(subset=[
    "hardness_score_smooth",
    "pseudo_mse_roll_median_60",
    "drilling_efficiency_roll_median_60",
    "formation_residual_roll_median_60",
    "speed",
]).copy()

segment_df = segment_df.sort_values(["well_id", "processing_time"]).copy()
segment_df["_row_in_well"] = segment_df.groupby("well_id").cumcount()
segment_df["segment_id"] = (segment_df["_row_in_well"] // SEGMENT_SIZE).astype(int)

segments = (
    segment_df
    .groupby(["well_id", "segment_id"])
    .agg(
        segment_start=("processing_time", "min"),
        segment_end=("processing_time", "max"),
        rows=("speed", "size"),
        hardness_segment=("hardness_score_smooth", "median"),
        pseudo_mse_segment=("pseudo_mse_roll_median_60", "median"),
        efficiency_segment=("drilling_efficiency_roll_median_60", "median"),
        residual_segment=("formation_residual_roll_median_60", "median"),
        speed_segment=("speed", "median"),
    )
    .reset_index()
)

segments["energy_type_segment_quantile"] = pd.qcut(
    segments["hardness_segment"],
    q=4,
    labels=energy_labels_4,
    duplicates="drop",
).astype(str)

segment_df = segment_df.merge(
    segments[["well_id", "segment_id", "hardness_segment", "energy_type_segment_quantile"]],
    on=["well_id", "segment_id"],
    how="left",
)

display(
    segments
    .groupby("energy_type_segment_quantile")
    .agg(
        segments=("segment_id", "size"),
        rows=("rows", "sum"),
        hardness_segment=("hardness_segment", "median"),
        pseudo_mse_segment=("pseudo_mse_segment", "median"),
        efficiency_segment=("efficiency_segment", "median"),
        residual_segment=("residual_segment", "median"),
        speed_segment=("speed_segment", "median"),
    )
    .sort_values("hardness_segment")
)

,segments,rows,hardness_segment,pseudo_mse_segment,efficiency_segment,residual_segment,speed_segment
energy_type_segment_quantile,,,,,,,
soft_low_energy,1801,94049,-2.186504,7.979763e+07,1.253212e-08,0.004020,0.01818
medium_low_energy,1800,96213,-0.362486,1.118751e+08,8.938281e-09,0.000371,0.01212
medium_high_energy,1800,95943,0.716003,1.369420e+08,7.304144e-09,-0.001799,0.01212
hard_high_energy,1801,96430,1.979057,1.865598e+08,5.363636e-09,-0.003793,0.00606


## 7. Возвращаем финальную разметку в основной датасет

In [8]:
merge_cols = [
    "processing_time",
    "well_id",
    "segment_id",
    "hardness_segment",
    "energy_type_segment_quantile",
]

df_out = df.merge(
    segment_df[merge_cols],
    on=["processing_time", "well_id"],
    how="left",
    suffixes=("", "_segment"),
)

df_out["rock_energy_type_final"] = df_out["energy_type_segment_quantile"]

display(df_out[[
    "processing_time",
    "well_id",
    "speed",
    "energy_input_proxy",
    "pseudo_mse",
    "drilling_efficiency",
    "formation_residual",
    "hardness_score_smooth",
    "segment_id",
    "hardness_segment",
    "rock_energy_type_final",
]].head())

,processing_time,well_id,speed,energy_input_proxy,pseudo_mse,drilling_efficiency,formation_residual,hardness_score_smooth,segment_id,hardness_segment,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,0.002755,338743.056,1.229314e+08,8.131666e-09,-0.005300,NaN,NaN,NaN,NaN
1,2025-08-24 10:10:00.260,19601,0.003030,279919.984,9.235235e+07,1.082452e-08,-0.005025,NaN,NaN,NaN,NaN
2,2025-08-24 10:10:05.199,19601,0.006060,324846.384,5.359617e+07,1.865497e-08,-0.001995,NaN,NaN,NaN,NaN
3,2025-08-24 10:10:14.610,19601,0.002755,274750.210,9.970810e+07,1.002564e-08,-0.005300,NaN,NaN,NaN,NaN
4,2025-08-24 10:10:34.197,19601,0.001515,289195.048,1.907619e+08,5.238679e-09,-0.006540,NaN,NaN,NaN,NaN


## 8. Сводка финальной разметки

In [9]:
summary_final = (
    df_out
    .groupby("rock_energy_type_final")
    .agg(
        rows=("speed", "size"),
        speed_median=("speed", "median"),
        pseudo_mse_smooth=("pseudo_mse_roll_median_60", "median"),
        efficiency_smooth=("drilling_efficiency_roll_median_60", "median"),
        residual_smooth=("formation_residual_roll_median_60", "median"),
        hardness_smooth=("hardness_score_smooth", "median"),
        pressure_axis_median=("pressure_axis", "median"),
        pressure_rotation_median=("pressure_rotation", "median"),
        rotation_median=("rotation", "median"),
    )
    .sort_values("hardness_smooth")
)

display(summary_final)

,rows,speed_median,pseudo_mse_smooth,efficiency_smooth,residual_smooth,hardness_smooth,pressure_axis_median,pressure_rotation_median,rotation_median
rock_energy_type_final,,,,,,,,,
soft_low_energy,94049,0.01818,7.974684e+07,1.254017e-08,0.003969,-2.194668,17867.0,14759.0,103.308
medium_low_energy,96213,0.01212,1.116768e+08,8.954801e-09,0.000290,-0.354254,19606.0,15166.0,103.008
medium_high_energy,95943,0.01212,1.369806e+08,7.300751e-09,-0.001792,0.714666,20037.0,15180.0,102.966
hard_high_energy,96430,0.00606,1.858249e+08,5.388009e-09,-0.003750,1.983451,19231.0,14404.0,103.410


## 9. Plotly surface для одной выбранной энергоёмкости

Построение поверхности `pressure_axis × pressure_rotation → speed`.

Важно: поверхность строится через surrogate-модель внутри выбранного класса энергоёмкости.
Текущий `rotation` и `hardness_score_smooth` фиксируются медианами выбранного класса.

In [13]:
surface_features = [
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "hardness_score_smooth",
    "pressure_balance",
    "axis_over_rot_pressure",
    "rot_pressure_over_axis",
    "rotation_efficiency",
    "energy_input_proxy",
]

SURFACE_HTML_DIR = Path("plotly_surfaces_html")
SURFACE_HTML_DIR.mkdir(exist_ok=True)

SURFACE_TYPE = "hard_high_energy"

surface_train = df_out[df_out["rock_energy_type_final"] == SURFACE_TYPE].dropna(subset=[
    "pressure_axis", "pressure_rotation", "rotation", "hardness_score_smooth", "speed"
]).copy()

print("Surface type:", SURFACE_TYPE)
print("Rows:", len(surface_train))

surface_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    ))
])

surface_model.fit(surface_train[surface_features], surface_train["speed"])

fixed_state = surface_train[surface_features].median()

p_ax_grid = np.linspace(
    surface_train["pressure_axis"].quantile(0.05),
    surface_train["pressure_axis"].quantile(0.95),
    60,
)

p_rot_grid = np.linspace(
    surface_train["pressure_rotation"].quantile(0.05),
    surface_train["pressure_rotation"].quantile(0.95),
    60,
)

PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

grid = pd.DataFrame({
    "pressure_axis": PA.ravel(),
    "pressure_rotation": PR.ravel(),
})

for col in surface_features:
    if col not in grid.columns:
        grid[col] = fixed_state[col]

grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

Z = surface_model.predict(grid[surface_features]).reshape(PA.shape)

fig = go.Figure()

fig.add_trace(go.Surface(
    x=PA,
    y=PR,
    z=Z,
    colorscale="Viridis",
    opacity=0.95,
    showscale=True,
))

fig.update_layout(
    title=f"Surface: pressure_axis × pressure_rotation → speed | rock_energy_type_final={SURFACE_TYPE}",
    scene=dict(
        xaxis_title="pressure_axis",
        yaxis_title="pressure_rotation",
        zaxis_title="speed",
    ),
    height=800,
)

single_surface_html = SURFACE_HTML_DIR / f"surface_{SURFACE_TYPE}.html"
fig.write_html(single_surface_html, include_plotlyjs=True, full_html=True)
print("Saved browser surface:", single_surface_html)

fig.show()

Surface type: hard_high_energy
Rows: 96430
Saved browser surface: plotly_surfaces_html\surface_hard_high_energy.html


## 10. Plotly surfaces для всех энергоёмкостей

In [14]:
SURFACE_HTML_DIR = Path("plotly_surfaces_html")
SURFACE_HTML_DIR.mkdir(exist_ok=True)

for surface_type in energy_labels_4:
    surface_train = df_out[df_out["rock_energy_type_final"] == surface_type].dropna(subset=[
        "pressure_axis", "pressure_rotation", "rotation", "hardness_score_smooth", "speed"
    ]).copy()

    if len(surface_train) < 500:
        print("Skip small class:", surface_type, len(surface_train))
        continue

    surface_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        ))
    ])

    surface_model.fit(surface_train[surface_features], surface_train["speed"])

    fixed_state = surface_train[surface_features].median()

    p_ax_grid = np.linspace(
        surface_train["pressure_axis"].quantile(0.05),
        surface_train["pressure_axis"].quantile(0.95),
        60,
    )

    p_rot_grid = np.linspace(
        surface_train["pressure_rotation"].quantile(0.05),
        surface_train["pressure_rotation"].quantile(0.95),
        60,
    )

    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({
        "pressure_axis": PA.ravel(),
        "pressure_rotation": PR.ravel(),
    })

    for col in surface_features:
        if col not in grid.columns:
            grid[col] = fixed_state[col]

    grid["pressure_balance"] = grid["pressure_axis"] / (grid["pressure_axis"] + grid["pressure_rotation"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]

    Z = surface_model.predict(grid[surface_features]).reshape(PA.shape)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=PA,
        y=PR,
        z=Z,
        colorscale="Viridis",
        opacity=0.95,
        showscale=True,
    ))

    fig.update_layout(
        title=f"Surface by energy type: {surface_type}",
        scene=dict(
            xaxis_title="pressure_axis",
            yaxis_title="pressure_rotation",
            zaxis_title="speed",
        ),
        height=800,
    )

    surface_html = SURFACE_HTML_DIR / f"surface_{surface_type}.html"
    fig.write_html(surface_html, include_plotlyjs=True, full_html=True)
    print("Saved browser surface:", surface_html)

    fig.show()

Saved browser surface: plotly_surfaces_html\surface_soft_low_energy.html


Saved browser surface: plotly_surfaces_html\surface_medium_low_energy.html


Saved browser surface: plotly_surfaces_html\surface_medium_high_energy.html


Saved browser surface: plotly_surfaces_html\surface_hard_high_energy.html


## 11. Сохранение результата

In [15]:
OUTPUT_PATH = "united_rock_energy_segment_quantile.csv"
CONFIG_PATH = "rock_energy_segment_quantile_config.json"

df_out.to_csv(OUTPUT_PATH, index=False)

config = {
    "data_path": DATA_PATH,
    "output_path": OUTPUT_PATH,
    "final_method": "energy_type_segment_quantile",
    "segment_size": SEGMENT_SIZE,
    "labels": energy_labels_4,
    "control_features_for_residual": control_features,
    "surface_features": surface_features,
    "residual_model_metrics": {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    },
    "interpretation": {
        "hardness_score_smooth": "continuous operational drilling resistance index",
        "rock_energy_type_final": "final discrete energy-response regime from segment-level quantile segmentation",
        "pseudo_mse": "proxy energy per penetration, not physical MSE",
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_PATH)
print("Saved:", CONFIG_PATH)

if HAS_JOBLIB:
    ARTIFACT_DIR = Path("rock_energy_segment_quantile_artifacts")
    ARTIFACT_DIR.mkdir(exist_ok=True)

    joblib.dump(expected_speed_model, ARTIFACT_DIR / "expected_speed_from_controls_model.joblib")
    print("Saved artifacts to:", ARTIFACT_DIR)

Saved: united_rock_energy_segment_quantile.csv
Saved: rock_energy_segment_quantile_config.json
Saved artifacts to: rock_energy_segment_quantile_artifacts
